# https://learn.microsoft.com/en-us/azure/azure-sql/database/azure-sql-python-quickstart?view=azuresql&tabs=windows%2Csql-auth
import os
import pyodbc, struct

from dotenv import load_dotenv
load_dotenv()
connection_string = f'DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={os.environ["SERVER"]};DATABASE={os.environ["DATABASE"]};UID={os.environ["UID"]};PWD={os.environ["PWD"]}'

def get_conn():
    conn = pyodbc.connect(connection_string)
    return conn

with get_conn() as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM dbo.Chart;")

    row = cursor.fetchone()
    print(row)



In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()

from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url_construct = URL.create(
    "mssql+pyodbc",
    username=os.environ["UID"],
    password=os.environ["PWD"],
    host=os.environ["SERVER"],
    database=os.environ["DATABASE"],
    query={"driver": "ODBC Driver 18 for SQL Server"}
)

engine = create_engine(url_construct, fast_executemany=True)

In [ ]:
from sqlalchemy import text
from sqlalchemy import inspect

insp = inspect(engine)
for t in insp.get_table_names():
    print(t)
    for c in insp.get_columns(t):
        print(f'\t{c}')

with engine.connect() as connection:
    result = connection.execute(text("select * from dbo.Chart"))
    for row in result:
        print(row)

In [ ]:
from datetime import datetime as dt, timezone as tz
from sqlalchemy import insert, delete
from sqlalchemy import MetaData

md = MetaData()
md.reflect(bind=engine)
catalog_table = md.tables['Catalog']

feb29 = dt(
    year=2024,
    month=2,
    day=29,
    tzinfo=tz.utc
)

"""
with engine.connect() as connection:
    result = connection.execute(
        insert(catalog_table),
        [
            {"catalog_id": 2, "name": "ITL2022", "active": False, "last_update": feb29},
            {"catalog_id": 3, "name": "ITL2023", "active": True, "last_update": feb29},
            {"catalog_id": 4, "name": "ITL2024", "active": True, "last_update": feb29},
            {"catalog_id": 1, "name": "StepManiaX", "active": True, "last_update": feb29}
        ],
    )
    # connection.commit()
"""

In [ ]:
from dataclasses import dataclass

# Database enhanced classes
@dataclass
class Catalog:
    catalog_id: int
    name: str
    active: bool
    last_update: dt

@dataclass
class Chart:
    global_chart_id: int
    catalog_id: int
    chart_id: int
    hash: int
    title: str
    subtitle: str
    artist: str
    meter: float
    slot: str
    style: str
    value: float
    value_scoring: float
    value_passing: float
    spice: float
    spice_calc_time: dt

@dataclass
class Player:
    performance_id: int
    catalog_id: int
    entrant_id: int
    groovestats_id: int
    boogiestats_id: int
    name: str
    scobility: float
    timing_power: float
    comfort_zone: float
    scobility_calc_time: dt

@dataclass
class Relationship:
    relationship_id: int
    x_id: int
    y_id: int
    common: int
    relation: float
    strength: float

@dataclass
class Score:
    score_id: int
    catalog_id: int
    global_chart_id: int
    performance_id: int
    plays: int
    last_played: dt
    clear: str
    score: float
    prediction: float
    prediction_calc_time: dt

In [ ]:
import json

catalog_itl2023 = {}
with open(r"C:\Users\telpi\Documents\GitHub\scobility\scratch\scobility_itl2023_20230926.json", 'r') as fp:
    catalog_itl2023 = json.load(fp)
catalog_itl2023["timestamp"] = dt(
    year=2023,
    month=9,
    day=26,
    tzinfo=tz.utc
)

catalog_itl2024 = {}
src_itl2024 = r"C:\Users\telpi\Documents\GitHub\scobility\scratch\scobility_itl2024_20240610.json"
with open(src_itl2024, 'r') as fp:
    catalog_itl2024 = json.load(fp)
catalog_itl2024["timestamp"] = dt.fromtimestamp(os.path.getmtime(src_itl2024), tz.utc)

catalog_itl2025 = {}
src_itl2025 = r"C:\Users\telpi\Documents\GitHub\scobility\scratch\scobility_itl2025_20250414.json"
with open(src_itl2025, 'r') as fp:
    catalog_itl2025 = json.load(fp)
catalog_itl2025["timestamp"] = dt.fromtimestamp(os.path.getmtime(src_itl2025), tz.utc)

In [ ]:
class Passthrough:
    def __init__(self, d):
        self.d = d
    
    def __delitem__(self, key):
        pass
    def __getitem__(self, key):
        if key in self.d:
            return self.d[key]
        else:
            return key
    def __setitem__(self, key):
        pass

remaps = {
    'charts': Passthrough({
        's_id': 'chart_id'
    }),
    'players': Passthrough({
        'g_id': 'groovestats_id',
        'e_id': 'entrant_id'
    }),
    'scores': Passthrough({
        's_id': 'global_chart_id',
        'e_id': 'performance_id',
        'value': 'score'
    })
}


In [ ]:
charts = []
players = []
scores = []

catalogs = {
    'ITL2023': {
        'data': catalog_itl2023,
        'catalog_id': 3,
        'chart_offset': 3000,
        'score_offset': 3000000,
        'performance_offset': 30000,
    },
    'ITL2024': {
        'data': catalog_itl2024,
        'catalog_id': 4,
        'chart_offset': 4000,
        'score_offset': 4000000,
        'performance_offset': 40000,
    },
    'ITL2025': {
        'data': catalog_itl2025,
        'catalog_id': 5,
        'chart_offset': 5000,
        'score_offset': 5000000,
        'performance_offset': 50000,
    },
}

catalog_choice = catalogs['ITL2025']

for i, s in enumerate(catalog_choice['data']['songs']):
    c = {remaps['charts'][k]: v for k, v in s.items()}
    c['style'] = 'dance-single'
    c['spice_calc_time'] = catalog_choice['data']['timestamp']
    c['global_chart_id'] = catalog_choice['chart_offset'] + c['chart_id']
    c['catalog_id'] = catalog_choice['catalog_id']
    charts.append(Chart(**c))

for i, q in enumerate(catalog_choice['data']['players']):
    p = {remaps['players'][k]: v for k, v in q.items()}
    p['boogiestats_id'] = None
    p['scobility_calc_time'] = catalog_choice['data']['timestamp']
    p['catalog_id'] = catalog_choice['catalog_id']
    p['performance_id'] = catalog_choice['performance_offset'] + p['entrant_id']
    del p['tourney_power']
    players.append(Player(**p))

for i, w in enumerate(catalog_choice['data']['scores']):
    s = {remaps['scores'][k]: v for k, v in w.items()}
    s['score'] = 1 - w['value']
    s['last_played'] = dt.strptime(w['last_played'], "%Y-%m-%dT%H:%M:%S.%f")
    s['prediction'] = 1
    s['prediction_calc_time'] = catalog_choice['data']['timestamp']
    s['catalog_id'] = catalog_choice['catalog_id']
    s['performance_id'] = catalog_choice['performance_offset'] + w['e_id']
    s['score_id'] = catalog_choice['score_offset'] + i
    s['global_chart_id'] += catalog_choice['chart_offset']
    scores.append(Score(**s))

In [ ]:
len(scores)

In [ ]:
performance_ids = set(p.performance_id for p in players)
scores = [s for s in scores if s.performance_id in performance_ids]

# set(p.performance_id for p in players) - set(s.performance_id for s in scores)
set(s.performance_id for s in scores) - set(p.performance_id for p in players)

In [ ]:
from dataclasses import asdict
import re

block_size = 300

In [ ]:
with engine.connect() as connection:
    try:
        delete_statement = delete(md.tables['Score']).where(md.tables['Score'].c.catalog_id == 5)
        connection.execute(delete_statement)
    except Exception as e:
        print(e)

In [ ]:
with engine.connect() as connection:
    result = connection.execute(
        delete(md.tables['Player']).where(
            md.tables['Player'].c.catalog_id == 5
        )
    )
    result = connection.execute(
        delete(md.tables['Chart']).where(
            md.tables['Chart'].c.catalog_id == 5
        )
    )
for i in range(len(players) // block_size + 1):
    with engine.connect() as connection:
        try:
            result = connection.execute(
                insert(md.tables['Player']),
                [
                    asdict(p) for p in players[(block_size*i):(block_size*(i+1))]
                ],
            )
            print(f"{(block_size*i):6d}~{((block_size*(i+1))-1):6d} added")
        except Exception as e:
            print(e)
for i in range(len(charts) // block_size + 1):
    with engine.connect() as connection:
        try:
            result = connection.execute(
                insert(md.tables['Chart']),
                [
                    asdict(c) for c in charts[(block_size*i):(block_size*(i+1))]
                ],
            )
            print(f"{(block_size*i):6d}~{((block_size*(i+1))-1):6d} added")
        except Exception as e:
            print(e)

In [ ]:

for i in range(len(scores) // block_size + 1):
    with engine.connect() as connection:
        try:
            connection.execute(insert(md.tables['Score']),
                [
                    asdict(s) for s in scores[(block_size*i):(block_size*(i+1))]
                ]
            )
            print(f"{(block_size*i):6d}~{((block_size*(i+1))-1):6d} added")
        except Exception as e:
            print(e)
        # connection.commit()
    #break